# Line Following Robot using IR Sensors
**Full Coding available at [Line_Following.py](Line_Following.py)** 
<br>
This is the documentation for line following using infrared (IR) sensors (Maker Line sensor array).

Line following is a fundamental robotics application where a robot follows a predetermined path marked by a line (usually black) on a contrasting surface (usually white). This implementation uses a 5-sensor IR array for precise line detection and navigation.

***Libraries Used In The Script***
- time - For timing operations and delays  
- RPi_Robot_Hat_Lib - Custom robot control library with line sensor support

## Installing Libraries & Dependencies 

- **Built-in Libraries**:
    - time (Built-in Python library)

- **Custom Libraries**:
    - RPi_Robot_Hat_Lib (Custom robot control library)

**Installation**: The RPi_Robot_Hat_Lib should be installed according to your robot hardware setup instructions.

## Let's Start Coding ! 
### 1. Import Required Libraries 
Import the necessary libraries for robot control and line following:

- **RPi_Robot_Hat_Lib**: Provides RobotController class for motor control and sensor reading
- **time**: Used for timing operations (though not actively used in this implementation)

In [ ]:
from RPi_Robot_Hat_Lib import RobotController 
import time 


### 2. The Initialization of `RobotController`

This step creates an instance of the `RobotController` class, which manages robot movement and sensor readings.

- **Function Name**: `RobotController`

- **Returns**:  
  - `RobotController` → The controller object used to control motors and read sensors  

- **Usage**:  
  - Create the controller instance:  
    ```python
    Motor = RobotController()
    ```
  - Provides methods for:  
    - Reading line sensor data  
    - Controlling motor movements (`Forward`, `Brake`, `move`)  
    - Managing cleanup and safe shutdown


In [ ]:
Motor = RobotController()

### 3. The Function `main()`

This function implements the **line following logic** using input from the robot’s line sensors and controls the robot’s motors accordingly.

- **Function Name**: `main`

- **Parameters**:  
  - *None*

- **Returns**:  
  - *None*

- **Global Variables**:  
  - `Motor` → Instance of `RobotController` used to control motors and read sensors  

- **Usage**:  
  - Continuously reads the **5-bit line sensor value**:  
    ```python
    Line_Sensor = Motor.read_line_sensors()
    ```
  - Each bit in the integer represents the **state of one sensor** (1 = line detected, 0 = no line).  
    Sensors (MSB → LSB):  
    ```
    outerRight | Right | Center | Left | outerLeft
    ```
  - The **bit shift (`>>`)** and **bitwise AND (`&`)** operations are used to isolate each sensor’s value:  
    ```python
    outerRight = (Line_Sensor >> 4) & 1  # Extract 5th bit
    Right      = (Line_Sensor >> 3) & 1  # Extract 4th bit
    center     = (Line_Sensor >> 2) & 1  # Extract 3rd bit
    Left       = (Line_Sensor >> 1) & 1  # Extract 2nd bit
    outerLeft  = Line_Sensor & 1         # Extract 1st bit
    ```
    - `>> n` shifts the binary number `n` positions to the right  
    - `& 1` masks all bits except the last one  
    - Example:  
      ```
      Line_Sensor = 0b10100  (binary for 20)
      outerRight = (0b10100 >> 4) & 1 = 1
      Right      = (0b10100 >> 3) & 1 = 0
      center     = (0b10100 >> 2) & 1 = 1
      Left       = (0b10100 >> 1) & 1 = 0
      outerLeft  = (0b10100) & 1      = 0
      ```
      Meaning → Outer Right and Center sensors detect the line.
  
  - Makes decisions based on sensor states:  
    - **All sensors 0** → Stop the robot  
      ```python
      Motor.Brake()
      ```
    - **Right sensors active** (`outerRight` or `Right`) → Turn Right  
      ```python
      Motor.move(speed=0, turn=30)
      ```
    - **Left sensors active** (`outerLeft` or `Left`) → Turn Left  
      ```python
      Motor.move(speed=0, turn=-30)
      ```
    - **Center aligned (with or without side support)** → Move Forward  
      ```python
      Motor.Forward(40)
      ```
    - **All sensors active** → Stop (junction/end detected)  
      ```python
      Motor.Brake()
      ```


In [ ]:
def main():
    while True: 
        Line_Sensor = Motor.read_line_sensors()
        outerRight = (Line_Sensor >> 4) & 1
        Right = (Line_Sensor >> 3) & 1
        center = (Line_Sensor >> 2) & 1
        Left = (Line_Sensor >> 1) & 1
        outerLeft = Line_Sensor & 1
        if outerRight == 0 and Right == 0 and center == 0 and Left == 0 and outerLeft == 0:
            Motor.Brake()
            print("Brake")

        if (outerRight == 1 or Right == 1) and center == 0 and Left == 0 and outerLeft == 0:
            print("Turn Right")  # Normal clockwise rotation
            Motor.move(speed=0, turn=30)

        if (outerLeft == 1 or Left == 1) and center == 0 and Right == 0 and outerRight == 0:
            Motor.move(speed=0, turn=-30)
            print("Turn Left")  # Normal anticlockwise rotation

        if (center == 1 and Right == 1 and Left == 1 and outerLeft == 0 and outerRight == 0) or (center == 1 and Right == 0 and Left == 0 and outerRight == 0 and outerLeft == 0):
            Motor.Forward(40)
            print("Forward")

        if center == 1 and Right == 1 and Left == 1 and outerRight == 1 and outerLeft == 1:
            Motor.Brake()
            print("Brake")


### 4. Script Execution and Interrupt Handling

This section ensures that the program starts correctly and exits gracefully when interrupted.

- **Execution Block**:  
  - Runs the `main()` function only if the script is executed directly.  
    ```python
    if __name__ == '__main__': 
        main()
    ```

- **Exception Handling**:  
  - **Keyboard Interrupt** (`Ctrl + C` in terminal):  
    - Stops the robot and cleans up the `RobotController` resources.  
    - Displays a message indicating program termination.  
        ```python
        except KeyboardInterrupt: 
            Motor.cleanup()
            print("Program Stopped")
        ```


In [ ]:
try: 
    if __name__ == '__main__': 
        main() 

except KeyboardInterrupt: 
    Motor.cleanup()
    print("Program Stopped")
